In [1]:
pip install -U sentence-transformers

     |████████████████████████████████| 85 kB 2.9 MB/s 
     |████████████████████████████████| 2.8 MB 24.2 MB/s 
     |████████████████████████████████| 1.2 MB 57.0 MB/s 
     |████████████████████████████████| 52 kB 2.1 MB/s 
     |████████████████████████████████| 3.3 MB 50.7 MB/s 
     |████████████████████████████████| 636 kB 61.4 MB/s 
     |████████████████████████████████| 895 kB 61.9 MB/s 
  Created wheel for sentence-transformers: filename=sentence_transformers-2.0.0-py3-none-any.whl size=126710 sha256=9b573f615dc7cd9e716e7032585760cb970d9b14a0a640f100f237f3bce72af8
  Stored in directory: /root/.cache/pip/wheels/d1/c1/0f/faafd427f705c4b012274ba60d9a91d75830306811e1355293
Successfully built sentence-transformers
  Attempting uninstall: pyyaml
    Found existing installation: PyYAML 3.13
    Uninstalling PyYAML-3.13:
      Successfully uninstalled PyYAML-3.13


In [2]:
pip install transformers

In [3]:
pip install datasets

     |████████████████████████████████| 270 kB 4.1 MB/s 
     |████████████████████████████████| 119 kB 63.3 MB/s 
     |████████████████████████████████| 243 kB 49.8 MB/s 
     |████████████████████████████████| 1.3 MB 65.5 MB/s 
     |████████████████████████████████| 142 kB 65.2 MB/s 
     |████████████████████████████████| 294 kB 65.9 MB/s 


In [4]:
pip install tqdm

In [14]:
import torch
import os
from sentence_transformers import SentenceTransformer, util
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import pipeline
from tqdm.notebook import tqdm
from datasets import load_dataset,concatenate_datasets


DATASET_NAME ='imdb'
LABELED = False
MAX_LENGTH = 120


def prepare_dataset(name):
    # train_ds = load_dataset(name, split='train[:80%]')
    # val_ds = load_dataset(name, split='train[80%:]')
    test_ds = load_dataset(name, split='test')
    train_ds = load_dataset(name, split='train')
    # train_ds = train_ds.map(lambda examples: {'labels': examples['label']})
    # val_ds = val_ds.map(lambda examples: {'labels': examples['label']})
    # test_ds = test_ds.map(lambda examples: {'labels': examples['label']})

    # train_ds = train_ds.train_test_split(100)  # for experiment we only use 500 examples
    # train_ds = train_ds['test']

    # test_ds = test_ds.train_test_split(100)  # for experiment we only use 500 examples
    # test_ds = test_ds['test']

    return train_ds, None, test_ds

In [6]:
embedder = SentenceTransformer('roberta-large')

train_ds, val_ds, test_ds = prepare_dataset(DATASET_NAME) # train_ds used as pool of examples, test_ds as the target dataset
class_names = test_ds.features["label"].names

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained('gpt2')

# prepocess the dataset
train_ds = train_ds.map(lambda e: {'text' : tokenizer.decode(tokenizer.encode(e['text'], padding = True, truncation= True, max_length = MAX_LENGTH), skip_special_tokens= True)})
test_ds = test_ds.map(lambda e: {'text' : tokenizer.decode(tokenizer.encode(e['text'], padding = True, truncation= True, max_length = MAX_LENGTH), skip_special_tokens= True)})


Downloading:   0%|          | 0.00/391 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/9.29k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/482 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/456k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/1.43G [00:00<?, ?B/s]

...

Downloading:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/665 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/548M [00:00<?, ?B/s]

  0%|          | 0/25000 [00:00<?, ?ex/s]

  0%|          | 0/25000 [00:00<?, ?ex/s]

In [7]:
print(len(train_ds))
print(len(test_ds))

25000
25000


In [8]:
# preprocess examples in train_ds as form of e = (sentence, label, embedding)
examples = [{'text': e['text'], 'label': e['label'],
              'embedding': embedder.encode(e['text'], convert_to_tensor=False)} for e in tqdm(train_ds)]
corpus_embeddings = torch.FloatTensor([example['embedding'] for example in examples])

top_k = min(3, len(examples))

  0%|          | 0/25000 [00:00<?, ?it/s]

In [10]:
def label2name(label):
    if DATASET_NAME == 'imdb':
      if label == 1:
          return 'good'
      else:
          return 'bad'
    elif DATASET_NAME == 'ag_news':
      return class_names[label]
    else: 
      return 'null'


def generate_prompt(indices, query, Labeled=True):
    prompt = ''

    if Labeled:
      for id in indices:
        name = label2name(examples[id]['label'])
        if DATASET_NAME == 'imdb':
          prompt += examples[id]['text'] + ' The movie is ' + name + '.\n'
        elif DATASET_NAME == 'ag_news':
          if name == 'Sci/Tech':
            name = 'Science and Technology'
          prompt += examples[id]['text'] + ' The article is about ' + name + '.\n'
        else:
          raise Exception('Unsupported Dataset')
    else: # unlabeled Priming
      for id in indices:
        if DATASET_NAME == 'imdb':
          tmp = examples[id]['text'] + ' The movie is'
          predicted = predict(tmp)
          prompt += examples[id]['text'] + ' The movie is ' + predicted + '.\n'
        elif DATASET_NAME == 'ag_news':
          tmp = examples[id]['text'] + ' The article is about'
          predicted = predict(tmp)
          if predicted == 'Science' or predicted == 'Technology':
            predicted = 'Science and Technology'
          prompt += examples[id]['text'] + ' The article is about ' + predicted + '.\n'
        else:
          raise Exception('Unsupported Dataset')
        

    if DATASET_NAME == 'imdb':
      prompt += query + ' The movie is'
    elif DATASET_NAME == 'ag_news':
      prompt += query + ' The article is about'
    else:
      raise Exception('Unsupported Dataset')

    return prompt

In [11]:
def predict(prompt:str,**kwargs):
    input_ids = tokenizer(prompt, return_tensors='pt')

    outputs = model(**input_ids)
    logits = outputs.logits
    if DATASET_NAME =='imdb':
      good_id = tokenizer.convert_tokens_to_ids('good')
      bad_id = tokenizer.convert_tokens_to_ids('bad')
      if logits[0][-1][good_id] > logits[0][-1][bad_id]:
        return 'good'
      else: 
        return 'bad'
    else:
      raise Exception('Unsupported Dataset')


In [16]:
_# iterate over the whole test_dataset
correct_num = 0
total_num = 0
test_ds = test_ds.train_test_split(1000)  # for experiment we only use 500 examples
test_ds = test_ds['test']
pbar = tqdm(test_ds, desc='Running')
corpus_embeddings = corpus_embeddings.to(device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'))

for query in pbar:
    query_embedding = embedder.encode(query['text'], convert_to_tensor=True)

    cos_scores = util.pytorch_cos_sim(query_embedding, corpus_embeddings)[0]
    top_results = torch.topk(cos_scores, k=top_k)
    indices = top_results[1].tolist()
    indices.reverse()  # we put the more similar example, nearer to our query

    prompt = generate_prompt(indices, query['text'], Labeled=LABELED)
    

    if DATASET_NAME == 'imdb':
      true_name = 'good' if query['label'] ==1 else 'bad'
    elif  DATASET_NAME == 'ag_news':
      true_name = class_names[query['label']]
    else:
      raise Exception('Unsupported Dataset')

    predicted = predict(prompt)
    # print(predicted)
    # print(true_name)
    if predicted!='good' and predicted!='bad':
      raise Exception('Unexpected Prediction')

    if true_name == 'Sci/Tech' and (predicted=='Science' or predicted=='Technology'):
      correct_num += 1
    elif predicted == true_name:
      correct_num += 1
    total_num += 1
    pbar.set_postfix({' binary_accuracy': correct_num/total_num})

print('%d/%d  accuracy: %.4f'%(correct_num, total_num, correct_num/total_num))

Running:   0%|          | 0/1000 [00:00<?, ?it/s]

649/1000  accuracy: 0.6490
